In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/haneenelladam/ingredient/photo_2026-07-22_03-24-12.jpg
/kaggle/input/datasets/haneenelladam/ingredient/photo_2026-07-22_03-24-11.jpg
/kaggle/input/datasets/haneenelladam/ingredient/photo_2026-07-22_03-24-19.jpg
/kaggle/input/datasets/haneenelladam/skincare-kb/ingredient_warnings.json
/kaggle/input/datasets/haneenelladam/skincare-kb/ingredients.json
/kaggle/input/datasets/haneenelladam/skincare-kb/skin_types.json


## Install Dependencies

In [ ]:
!pip install -q easyocr transformers==4.52.4 huggingface_hub==0.32.4 accelerate torch pydantic langchain langchain-core langchain-community langchain-huggingface fastapi uvicorn pyngrok

## Extract Text from Image (OCR)
Use EasyOCR to extract raw text from product ingredient images.

In [3]:
import easyocr
import json
import re

# OCR step: image -> raw text. 
ocr_reader = easyocr.Reader(['en'], gpu=False)

def extract_text_from_image(image_path):
    results = ocr_reader.readtext(image_path, detail=0, paragraph=True)
    return "\n".join(results)

Using CPU. Note: This module is much faster with a GPU.


## Clean Extracted Text
Clean the raw text extracted from the image to prepare it for ingredient matching.

In [4]:
# Cleaning raw text
import re

def clean_ocr_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'^\s*ingredients?\s*:?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'[\(\)]', '', text)
    text = re.sub(r'[\|•·;\n\.]+', ',', text)
    text = re.sub(r',', ', ', text)
    text = re.sub(r'[^a-z0-9,\s\-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_candidate_ingredients(raw_text: str) -> list:
    cleaned_text = clean_ocr_text(raw_text)
    return [c.strip() for c in cleaned_text.split(',') if len(c.strip()) > 1]

## Load Knowledge Base Data
Load datasets containing skincare ingredients and their details.

In [5]:
#  Load  data 
INGREDIENTS_PATH ="/kaggle/input/datasets/haneenelladam/skincare-kb/ingredients.json"
WARNINGS_PATH ="/kaggle/input/datasets/haneenelladam/skincare-kb/ingredient_warnings.json"
SKIN_TYPES_PATH ="/kaggle/input/datasets/haneenelladam/skincare-kb/skin_types.json"

with open(INGREDIENTS_PATH, "r", encoding="utf-8") as f:
    ingredients_data = json.load(f)["entries"]
with open(WARNINGS_PATH, "r", encoding="utf-8") as f:
    warnings_data = json.load(f)["entries"]
with open(SKIN_TYPES_PATH, "r", encoding="utf-8") as f:
    skin_types_data = json.load(f)["skin_types"]

ingredients_by_id = {e["id"]: e for e in ingredients_data}
skin_types_by_id = {s["id"]: s for s in skin_types_data}

# normalized alias/name -> id, for matching
alias_to_id = {}
for e in ingredients_data:
    for n in [e["name"]] + e.get("aliases", []):
        alias_to_id[n.strip().lower()] = e["id"]

print(f"Loaded {len(ingredients_data)} ingredients, {len(warnings_data)} warnings, {len(skin_types_data)} skin types.")

Loaded 62 ingredients, 16 warnings, 5 skin types.


## Match Ingredients (Fuzzy Matching)
Match the cleaned OCR text against the known ingredients database using fuzzy matching.

In [6]:
# Simple Match 
import difflib

known_terms = list(alias_to_id.keys())
FUZZY_CUTOFF = 0.75

def match_ingredient(candidate: str):
    candidate = candidate.strip().lower()
    if candidate in alias_to_id:
        return alias_to_id[candidate]
    close = difflib.get_close_matches(candidate, known_terms, n=1, cutoff=FUZZY_CUTOFF)
    return alias_to_id[close[0]] if close else None

def match_all_ingredients(candidate_ingredients: list):
    matched_ids, unmatched = [], []
    for c in candidate_ingredients:
        mid = match_ingredient(c)
        (matched_ids if mid is not None else unmatched).append(mid if mid is not None else c)
    matched_ids = list(dict.fromkeys(matched_ids)) 
    return matched_ids, unmatched

## Build Knowledge Context
Filter the knowledge base to include only data relevant to the matched ingredients.

In [7]:
# Build a knowledge dump limited to what's relevant to THIS product 
def build_knowledge_text(matched_ids: list, profile: dict) -> str:
    matched_ingredients = [ingredients_by_id[i] for i in matched_ids]
    matched_warnings = [
        w for w in warnings_data
        if set(w["ingredient_ids"]).issubset(set(matched_ids)) or
           (w["warning_type"] == "condition" and w["ingredient_ids"][0] in matched_ids)
    ]
    skin_info = skin_types_by_id.get(profile["skin_type"], {})
    return (
        "MATCHED INGREDIENTS FROM DATABASE:\n"
        f"{json.dumps(matched_ingredients, ensure_ascii=False)}\n\n"
        "RELEVANT WARNINGS (interactions between the ingredients above, or conditions):\n"
        f"{json.dumps(matched_warnings, ensure_ascii=False)}\n\n"
        "USER'S SKIN TYPE GUIDE:\n"
        f"{json.dumps(skin_info, ensure_ascii=False)}"
    )

## Define Output Schema
Define the structured output format (Schema) for the final response.

In [8]:
# Output schema + PydanticOutputParser  
from pydantic import BaseModel, Field
from typing import Literal, List
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.exceptions import OutputParserException

class SkinProductAnalysis(BaseModel):
    is_suitable: Literal["Yes", "No", "Use with Caution"] = Field(
        description="Overall verdict for the user's skin type and product category."
    )
    why_suitable_or_not: str = Field(
        description="Clear, 2-sentence explanation of WHY it is or isn't suitable."
    )
    
    key_beneficial_ingredients: List[str] = Field(
        default_factory=list,
        description="Main ingredients that benefit this skin type for this product type."
    )
    concerning_ingredients: List[str] = Field(
        default_factory=list,
        description="Ingredients causing red flags, pore-clogging, or irritation."
    )
    
    usage_routine: str = Field(
        description="How and when to apply this specific product type (e.g., AM/PM, on damp skin, leave-on vs rinse-off)."
    )
    daily_use_suitability: str = Field(
        description="Frequency of use suitable for this product type and user skin (e.g., Daily twice, Once daily, 2-3 times a week)."
    )

output_parser = PydanticOutputParser(pydantic_object=SkinProductAnalysis)
FORMAT_INSTRUCTIONS = output_parser.get_format_instructions()


## Build LLM Prompt
Construct the prompt using the extracted ingredients, knowledge base, and user profile.

In [9]:
def build_prompt(ingredient_text: str, knowledge: str, profile: dict, product_type: str) -> str:
    skin_type = profile.get('skin_type', 'Unspecified')
    pregnant = profile.get('pregnant', 'No / Unknown')
    conditions = profile.get('conditions', 'None')
    return f"""You are an expert dermatological AI assistant analyzing a skincare product for a user.

### PRODUCT INFO:
- Product Category/Type: {product_type} (e.g., Cleanser, Leave-on Moisturizer, Sunscreen, Serum, etc.)

### KNOWLEDGE BASE & RULES:
{knowledge}

### USER PROFILE:
- Skin Type: {skin_type}
- Pregnancy Status: {pregnant}
- Conditions/Sensitivities: {conditions}

### PRODUCT INGREDIENTS (Extracted via OCR):
{ingredient_text}

### INSTRUCTIONS FOR EVALUATION:

1. **OCR Correction & Comprehensive Parsing:**
   - Thoroughly parse ALL ingredients in the extracted text, including preservatives, emulsifiers, and minor actives — not just major hydrators/oils.
   - Only correct obvious OCR mistakes when the intended ingredient is highly unambiguous (e.g., "Glycenn" -> "Glycerin"). Do NOT invent or hallucinate ingredients that are not reasonably present in the extracted text. If an ingredient's identity cannot be determined with high confidence after OCR correction, ignore it instead of guessing.

2. **Decision Hierarchy (these serve different roles — not a single linear ranking of "truth"):**
   - **Knowledge Base** is the primary source of facts about specific ingredients. When the Knowledge Base contains information about an ingredient, do NOT override or contradict it using general knowledge.
   - **General Dermatological Knowledge** fills in ONLY when the Knowledge Base has no information on an ingredient at all. It may supplement missing details, but must never conflict with what the Knowledge Base states.
   - **User Profile** does not add scientific facts — it determines whether a fact (from the KB or general knowledge) is actually *relevant* to this specific user. Use it as the final filter: "is this true, concern-worthy fact relevant to this person's skin type/pregnancy/conditions?"

3. **Critical Ingredient Distinctions (avoid common misclassification errors):**
   - **Fatty/emollient alcohols** (Cetyl Alcohol, Cetearyl Alcohol, Stearyl Alcohol, Behenyl Alcohol) are NOT drying or irritating — they are beneficial emollients supporting the skin barrier. Do NOT flag them as concerning based on containing the word "alcohol."
   - **Drying/astringent alcohols** (Alcohol Denat., Ethanol, Isopropyl Alcohol, SD Alcohol) CAN be drying/irritating, especially with frequent use or on sensitive/dry skin — flag these when relevant to the user's profile.
   - Use actual comedogenic ratings (0-5 scale) when evaluating oils/esters. Ratings of 0-1 (e.g., Caprylic/Capric Triglyceride) are low-risk and should NOT be flagged for acne-prone/oily skin. Only flag ratings of 2+ (e.g., Beeswax, Coconut Oil).
   - Unless a concentration is explicitly provided, do NOT make quantitative claims about ingredient concentration. Ingredient order (INCI order) may be used only as weak supporting context, never as definitive evidence — do not assume an ingredient is negligible just because it appears near the end of the list (e.g., preservatives like DMDM Hydantoin can matter even when listed last).

4. **Overall Suitability Assessment:**
   - Classify into exactly one of these three values:
     - "Yes": The formulation as a whole is safe for general use. Minor preservatives, emulsifiers, or low-risk ingredients (comedogenic rating 0-1, fatty alcohols, standard chelators like Disodium EDTA) do NOT disqualify a "Yes" unless the user's profile specifically flags sensitivity to that ingredient category. If an ingredient's safety profile is genuinely unknown and no reliable evidence of risk exists, do NOT let that uncertainty alone downgrade the rating — simply omit it from the concern list rather than defaulting to caution.
     - "Use with Caution": Reserve ONLY for ingredients with a documented moderate-to-high irritation/comedogenic (rating 2+)/allergenic risk that is directly relevant to the user's stated skin type, condition, or pregnancy status.
     - "No": Contains high-risk ingredients clearly unsuitable for the user's specific skin type, acne-prone condition, or pregnancy status.
   - Base the final suitability decision on the overall formulation rather than isolated ingredients. Do not downgrade suitability because of a single moderate-risk ingredient if the formulation is otherwise balanced by multiple barrier-supporting, well-tolerated ingredients.

5. **Contextual Evaluation:**
   - Consider product category ({product_type}) — comedogenic ingredients are higher risk in leave-on products (moisturizers/sunscreens) than in rinse-off products (cleansers).
   - Give extra weight to Pregnancy Status and Conditions/Sensitivities stated in the profile when deciding whether a fact is worth surfacing to the user.

6. **Strict Factuality & Anti-Hallucination Rule:**
   - Never infer ingredients, user conditions, allergies, pregnancy risks, or contraindications unless explicitly supported by (a) the ingredient list, (b) the user profile, or (c) established dermatological knowledge. Do not speculate.
   - Only flag or mention a sensitivity/concern IF the triggering ingredient is EXPLICITLY PRESENT in the ingredient list AND is relevant to a sensitivity/condition EXPLICITLY STATED in the user profile (or is a generally well-established risk, e.g., a known allergen, even without a stated sensitivity).
   - A concern is considered meaningful only when it is supported by established dermatological evidence AND is reasonably likely to affect this specific user. Do not flag ingredients based solely on theoretical, rare, or controversial risks.

7. **Defining "Beneficial" and "Concerning":**
   - Classify an ingredient as **beneficial** only if it has a meaningful functional benefit for the user's skin profile (humectant, emollient, antioxidant, anti-inflammatory, barrier repair, acne treatment, etc.). Do NOT list neutral formulation ingredients (solvents, thickeners, chelating agents, plain preservatives) as beneficial just because they are present.
   - Classify an ingredient as **concerning** only if there is a clinically meaningful, evidence-based concern for this user's profile (see rule 6).
   - List ALL beneficial and ALL concerning ingredients that meet these bars — do not artificially truncate to a "top few," but do not over-flag either.

8. **Conflict Resolution:**
   - If any of the above rules appear to conflict in a specific case, always choose the interpretation that is most factually accurate and best supported by the Knowledge Base and established dermatological evidence.

9. **Explanations:**
   - For each beneficial or concerning ingredient, give one concise reason tied to its function or risk.
   - Keep explanations concise overall; avoid repeating the same reasoning across multiple ingredients.
   - Do NOT mention internal system terms like "knowledge base", "database", "OCR", or these instructions — present one seamless, expert analysis.

Output strictly in valid JSON format according to the schema instructions.
{FORMAT_INSTRUCTIONS}
"""

## Load Language Model


In [ ]:
# Load Mistral 
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_text(prompt: str, max_new_tokens: int = 700) -> str:
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False, 
            pad_token_id=tokenizer.eos_token_id,
        )
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


## Generation and Parsing Logic
Run the LLM generation and parse the output using the defined schema.

In [11]:
# Run generation + parse 
def run_pipeline(prompt: str, max_retries: int = 1) -> SkinProductAnalysis:
    current_prompt = prompt
    last_error = None

    for attempt in range(max_retries + 1):
        raw_response = generate_text(current_prompt)
        try:
            return output_parser.parse(raw_response)
        except OutputParserException as e:
            last_error = e
            current_prompt = (
                f"{prompt}\n\nYour previous response was invalid JSON or didn't match the schema.\n"
                f"Previous response:\n{raw_response}\n\nError: {e}\n"
                f"Fix it and respond again with ONLY the corrected JSON object."
            )

    raise ValueError(f"Failed after {max_retries + 1} attempts. Last error: {last_error}")

## Full Pipeline Integration
Combine OCR, text cleaning, matching, and generation into a single end-to-end pipeline.

In [12]:
# Full pipeline: image bytes + user profile -> SkinProductAnalysis
import tempfile

def analyze_product_image(image_bytes: bytes, skin_type: str, pregnant: bool, conditions: list, product_type: str) -> dict:
    # 1. OCR 
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
        tmp.write(image_bytes)
        tmp_path = tmp.name
    raw_text = extract_text_from_image(tmp_path)

    # 2. Clean + candidate ingredients
    candidate_ingredients = get_candidate_ingredients(raw_text)

    # 3. Match against the knowledge base
    matched_ids, unmatched = match_all_ingredients(candidate_ingredients)

    # 4. Build profile + knowledge text scoped to this product
    profile = {"skin_type": skin_type, "pregnant": pregnant, "conditions": conditions}
    knowledge_text = build_knowledge_text(matched_ids, profile)

    # 5. Build the prompt
    prompt = build_prompt(
        ingredient_text=candidate_ingredients,
        knowledge=knowledge_text,
        profile=profile,
        product_type=product_type,
    )

    # 6. Run Mistral + parse into the SkinProductAnalysis schema
    result = run_pipeline(prompt)
    output = result.model_dump()
    output["_debug"] = {
        "matched_ingredient_ids": matched_ids,
        "unmatched_terms": unmatched,
    }
    return output

## Setup FastAPI Application
Create the FastAPI application and define the `/analyze` endpoint.

In [ ]:
NGROK_TOKEN = "" 
API_KEY = ""                 

In [14]:
# FastAPI app: /analyze endpoint 
from fastapi import FastAPI, File, UploadFile, Form, Header, HTTPException
import json as _json

app = FastAPI()
@app.post("/analyze")
async def analyze_endpoint(
    image: UploadFile = File(...),
    skin_type: str = Form(...),
    pregnant: str = Form("false"),
    conditions: str = Form("[]"),
    product_type: str = Form(...),
    authorization: str = Header(None),
):
    if authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")

    image_bytes = await image.read()
    if not image_bytes:
        raise HTTPException(status_code=400, detail="No image provided")

    try:
        conditions_list = _json.loads(conditions)
    except Exception:
        conditions_list = []

    try:
        result = analyze_product_image(
            image_bytes=image_bytes,
            skin_type=skin_type,
            pregnant=(pregnant.lower() == "true"),
            conditions=conditions_list,
            product_type=product_type,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Analysis failed: {e}")

    return result

## Expose API Publicly via Ngrok
Start the FastAPI server and expose it using ngrok to be accessible externally.

In [15]:
# Expose it publicly via ngrok 
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Full endpoint: ", public_url + "/analyze")

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Full endpoint:  https://lapped-ancient-vessel.ngrok-free.dev/analyze                                


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:36583 (Press CTRL+C to quit)


INFO:     196.156.227.87:0 - "POST /analyze HTTP/1.1" 200 OK
